# Lesson 14 · How strong is your prior?

**Source:** Beyond the book: combines Question 1.2.2 (Lesson 01) with Lesson 12 (Chapter 13 of *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017)). Second dataset: the rat tumor experiments from Gelman et al., *Bayesian Data Analysis*, Section 5.3.

**Question:** A small change in the spread you believe makes a big change in the prior's strength α + β. When can the data tell us the right strength, and what goes wrong when they can't?

**Goal:** Translate a prior's spread into its strength, see how strength controls shrinkage, measure how precisely data determine it (profile likelihood), and show what an over-strong prior does to credible intervals, in baseball and in a small real dataset.

**Contents**

1. Spread is strength
2. What strength does to an estimate
3. Let the data decide: baseball
4. When the data can't decide: 30 players
5. A second example: rat tumor experiments
6. What to do about it

**Tags:** `prior-strength` `sensitivity-analysis` `beta-binomial` `profile-likelihood` `sample-size` `calibration` `empirical-bayes` `rat-tumors`

**Before you start:** Lessons 01 and 02. Sections 1–2 need only Lesson 01; Sections 3–4 build on Lessons 02 and 12. A worked version of this lesson is in `14_how_strong_is_your_prior_article.ipynb`; try the prompts before reading it.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: the Lahman data

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (7, 4)

## 1. Spread is strength

With mean μ, strength ν = α + β and standard deviation σ:

$$\nu = \frac{\mu(1-\mu)}{\sigma^2} - 1, \qquad \alpha = \mu\nu, \qquad \beta = (1-\mu)\nu.$$

**14.1.1 · Your turn.** Write `strength_from_spread(mu, sigma)`. With μ = 0.27, make a table of ν, α and β for σ = 0.02, 0.0256, 0.035 and 0.05.

In [ ]:
# 14.1.1 · Your code here

> **Check:** σ = 0.0256 gives about Beta(81, 219), ν ≈ 300; σ = 0.02 gives ν ≈ 490; σ = 0.035 gives ν ≈ 160.

**14.1.2 · Your turn.** Plot the three priors for σ = 0.02, 0.0256 and 0.035 on one chart.

In [ ]:
# 14.1.2 · Your code here

**14.1.3 · Reflect.** The strength depends on $1/\sigma^2$. In words, what happens to the prior's strength if you halve the spread you believe? Why is that dangerous when σ is a guess?

*14.1.3 · Your answer:*

## 2. What strength does to an estimate

The posterior mean is a weighted average of the player's record and the prior mean:

$$\hat p = \frac{H + \mu\nu}{AB + \nu} = w\,\frac{H}{AB} + (1-w)\,\mu, \qquad w = \frac{AB}{AB+\nu}.$$

**14.2.1 · Your turn.** Plot the weight w against AB (log x-axis, AB from 1 to 20,000) for ν = 160, 300 and 490.

In [ ]:
# 14.2.1 · Your code here

**14.2.2 · Your turn.** For records 4/10, 100/300 and 3000/10000, tabulate the posterior mean under each of the three priors.

In [ ]:
# 14.2.2 · Your code here

> **Check:** for 100/300 the three estimates are about 0.311, 0.302 and 0.294.

**14.2.3 · Reflect.** For which players does the choice of ν matter most, and for which does it hardly matter? What does ν mean in at-bats?

*14.2.3 · Your answer:*

## 3. Let the data decide: baseball

Use `eb_data.career(pitcher_rule=None)` and, as in Lesson 02, fit on players with AB > 500. Write the beta-binomial with mean μ and strength ν: `stats.betabinom.logpmf(H, AB, mu * nu, (1 - mu) * nu)`.

**14.3.1 · Your turn.** Write `fit_mu_nu(H, AB)` that finds the maximum-likelihood μ and ν. Optimizing `logit(mu)` and `log(nu)` keeps them in range. Report μ, ν, α, β and the implied σ.

In [ ]:
# 14.3.1 · Your code here

> **Check:** with the 2025 Lahman data, μ ≈ 0.260 and ν ≈ 367 (about Beta(95, 271)), implied σ ≈ 0.023.

**14.3.2 · Your turn.** Write `profile_loglik(nu, H, AB)`: fix ν, maximize the log-likelihood over μ only (`optimize.minimize_scalar` with bounds), and return the best value. Evaluate it on a log-spaced grid of ν from 50 to 2,000, plot it relative to its maximum, and find the ν values within 1.92 of the best (an approximate 95% interval).

In [ ]:
# 14.3.2 · Your code here

> **Check:** the interval is narrow, roughly 350 to 385 (the exact ends depend on how fine your grid is).

**14.3.3 · Reflect.** Compare the data's interval for ν with the range your guesses in Section 1 covered. What does that say about guessing a prior versus estimating one here?

*14.3.3 · Your answer:*

## 4. When the data can't decide: 30 players

Treat your fitted prior as the truth. Simulate a league of `n_players` by drawing real career AB at random (`rng.choice(career["AB"], ...)`), true averages from the prior, and hits from the binomial.

**14.4.1 · Your turn.** Write `simulate(n_players)` returning hits, AB and the true p. For 30, 300 and 3,000 players, plot the profile likelihood of ν on one chart with the true ν marked.

In [ ]:
# 14.4.1 · Your code here

> **Check:** with 30 players the 95% range typically spans several-fold (for example ~200 to ~1,000); with 3,000 it is narrow.

**14.4.2 · Your turn.** For 30, 100, 300 and 1,000 players, run 100 replications each: fit μ and ν, compute 95% credible intervals for every player, and record the estimated ν and the coverage (fraction of intervals containing the true p). Print progress as you go, and summarize by size.

In [ ]:
# 14.4.2 · Your code here

> **Check:** mean coverage climbs from about 0.88 (30 players) to about 0.95 (1,000 players); the worst 30-player replication can be far below.

**14.4.3 · Your turn.** Plot (a) the estimated ν by size (box plots, log scale) and (b) coverage against estimated ν ÷ true ν, one color per size.

In [ ]:
# 14.4.3 · Your code here

**14.4.4 · Reflect.** Which mistake hurts coverage more: overestimating ν or underestimating it? Explain why in terms of shrinkage.

*14.4.4 · Your answer:*

## 5. A second example: rat tumor experiments

Seventy historical control groups of about 20 rats each, plus a current experiment with 4 tumors in 14 rats (Gelman et al., *Bayesian Data Analysis*, Section 5.3). The data:

```python
y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2,
              2, 2, 2, 1, 5, 2, 5, 3, 2, 7, 7, 3, 3, 2, 9, 10, 4, 4, 4, 4, 4, 4, 4, 10, 4, 4, 4, 5,
              11, 12, 5, 5, 6, 5, 6, 6, 6, 6, 16, 15, 15, 9, 4])
n = np.array([20, 20, 20, 20, 20, 20, 20, 19, 19, 19, 19, 18, 18, 17, 20, 20, 20, 20, 19, 19, 18, 18,
              25, 24, 23, 20, 20, 20, 20, 20, 20, 10, 49, 19, 46, 27, 17, 49, 47, 20, 20, 13, 48, 50,
              20, 20, 20, 20, 20, 20, 20, 48, 19, 19, 19, 22, 46, 49, 20, 20, 23, 19, 22, 20, 20, 20,
              52, 46, 47, 24, 14])
```

The last entry (4/14) is the current experiment; fit the prior on the first 70.

**14.5.1 · Your turn.** Fit μ and ν to the 70 historical experiments and plot the profile likelihood of ν (grid from 1 to 1,000).

In [ ]:
# 14.5.1 · Your code here

> **Check:** about Beta(2.3, 14.1) (ν ≈ 16), with a 95% range of roughly 9 to 33.

**14.5.2 · Your turn.** For the current experiment, compute the posterior mean and 95% interval using the weakest plausible, best and strongest plausible ν. Plot the three posteriors with the raw rate marked.

In [ ]:
# 14.5.2 · Your code here

> **Check:** posterior mean from about 0.23 (weakest) to 0.18 (strongest); the upper 95% limit from about 0.41 to 0.31.

**14.5.3 · Reflect.** Why is ν so much less certain here than in baseball? If you had to report on the current experiment, how would you present the result honestly?

*14.5.3 · Your answer:*

## 6. What to do about it

**14.6.1 · Reflect.** Write three or four rules of thumb for choosing or checking a prior's strength, drawing on this lesson. Where does a full Bayesian model (Lesson 13) fit in?

*14.6.1 · Your answer:*

**14.6.2 · Reflect.** **Card connection.** Collectors estimate each Pokémon card set's *pull rate* (the chance that one sealed booster pack contains a rare card, a *hit*) by logging the packs they open. Suppose you had pull rates for ~120 sets, each with a few dozen to a few hundred packs opened: is that closer to the baseball case or the rat case? What would you check before trusting an empirical Bayes prior?

*14.6.2 · Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapters 2–3 (the beta prior and its fit) and 13 (simulating replications, varying sample size).
- Gelman, Carlin, Stern, Dunson, Vehtari & Rubin, *Bayesian Data Analysis* (3rd ed.), Section 5.3: the rat tumor experiments.
- Tarone (1982), "The use of historical control information in testing for a trend in proportions", *Biometrics* 38(1): the source of the rat tumor data.
- Lahman Baseball Database (CC BY-SA 3.0): https://sabr.org/lahman-database/
- Lesson 01 (Question 1.2.2), Lesson 12 and Lesson 13 of this series.